# Fake News Detector — Colab Training

This notebook trains all model×embedding×dataset combinations on Google Colab,
persisting every artifact to Google Drive so nothing is lost on session timeout.

### Workflow
1. Mount Google Drive
2. Clone / pull the repo from GitHub
3. Symlink Drive folders ↔ project (datasets in, models out)
4. Install dependencies
5. Train per dataset

### First-time Drive setup
Upload these folders to `My Drive/fake-news-results/` once:
```
fake-news-results/
├── datasets/
│   ├── processed/      ← ISOT/, LIAR/, WELFake/ (train.csv, test.csv, val.csv)
│   ├── embeddings/     ← glove.6B.100d.txt, GoogleNews-vectors-negative300.bin
│   ├── web_scraped_data/
│   │   └── processed/     ← web_title.csv, web_short_text.csv, web_text.csv
```
Everything else (saved_models, experiments, mlflow.db, …) is created automatically.

## 0 — Configuration
Load configuration from colab_secrets.json (NOT hardcoded).
This keeps your GitHub token private and out of version control.

**First time setup:**
1. Create `colab_secrets.json` from `colab_secrets.json.template`
2. Fill in your GitHub personal access token
3. Keep `colab_secrets.json` in `.gitignore` (already added)

In [1]:
import sys
from pathlib import Path

# Import secure config loader
sys.path.insert(0, str(Path.cwd()))
from research.colab.colab_config import load_colab_config

# Load configuration from secure file
try:
    config = load_colab_config()
    REPO_URL = config['github']['repo_url']
    BRANCH = config['github']['branch']
    DRIVE_ROOT = config['colab']['drive_root']
    PROJECT_DIR = config['colab']['project_dir']
    print("[OK] Configuration loaded from colab_secrets.json")
except FileNotFoundError as e:
    print(f"[ERROR] {e}")
    print("\nPlease create colab_secrets.json first!")
    raise
except ValueError as e:
    print(f"[ERROR] {e}")
    raise

[OK] Configuration loaded from colab_secrets.json


## 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2 — Clone or update repository
First run clones; subsequent runs pull the latest changes.
After pushing code changes locally, just re-run this cell.

In [ ]:
import os

if os.path.isdir(PROJECT_DIR):
    print("Repo exists — pulling latest changes…")
    !cd {PROJECT_DIR} && git pull origin {BRANCH}
else:
    print("Cloning repo…")
    !git clone --branch {BRANCH} {REPO_URL} {PROJECT_DIR}

%cd {PROJECT_DIR}
!git log --oneline -3

## 3 — Symlink Drive ↔ project
Links datasets/embeddings **into** the project and output dirs **out** to Drive.

In [ ]:
!python -m research.colab.colab_setup --project {PROJECT_DIR} --drive {DRIVE_ROOT}

## 4 — Install dependencies

In [ ]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 46.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 120.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 6.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.4/117.4 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 239.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 125.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 19.9 MB/s eta 0:00:00

## 5 — Train models
Each cell trains one dataset. Re-runnable: Optuna studies use
`load_if_exists=True`, so interrupted runs resume where they left off.

Key Hydra overrides you can add:

- `datasets_list=[ISOT,LIAR]` — subset of datasets
- `models_to_optimize=[svm,xgb]` — subset of models
- `embeddings_to_use=[tfidf,glove]` — subset of embeddings
- `optuna.n_trials=20` — more tuning trials
- `hydra.job.chdir=False` — disables dynamic output folders to keep your Drive symlinks and relative paths intact

In [ ]:
!python -m research.train_models

## 6 — Collect web data
Scrapes real-world news articles, preprocesses with the same pipeline as training data.
Outputs three CSVs to `experiments/web_test_results/` (title, excerpt, full text) for
generalization testing on out-of-distribution data.

In [ ]:
!python -m research.web.collect_web

## 7 — Evaluate on web data
Runs all trained models against collected web data. Compares across datasets (ISOT, LIAR, WELFake),
model types (SVM, XGBoost, NN), embeddings (TF-IDF, GloVe, BERT), and text types (title, excerpt, full text).
Outputs predictions to `experiments/web_test_results/results_*.csv`.

In [ ]:
!python -m research.evaluate_web

## 8 — Analyze results
Aggregates all results into reports and visualizations. Computes F1/Precision/Recall across
all combinations, calculates generalization gaps, and generates charts saved to `presentation_charts/`.
Check `GENERALIZATION_SUMMARY.md` for a human-readable summary of top-performing models.

In [ ]:
!python -m research.analyze_results